# Ingestão de Dados | Camada Bronze

## Objetivo

Este notebook realiza a ingestão dos dados brutos do mercado imobiliário das cidades de Salvador, São Paulo, Rio de Janeiro, Recife e Curitiba.

Os arquivos CSV utilizados são provenientes da fonte pública Guru dos Imóveis e são carregados preservando os dados de origem. Durante a ingestão são adicionados metadados de identificação da cidade e da Unidade Federativa (UF), permitindo a rastreabilidade e posterior integração dos dados.

Os dados resultantes serão armazenados em formato Delta na camada Bronze do catálogo `mvp_imobiliario`.

## Arquitetura

**Fonte CSV → Camada Bronze → Camada Silver → Camada Gold**

Neste notebook é executada exclusivamente a etapa de ingestão para a camada Bronze. Os processos de limpeza, padronização, validação e modelagem serão realizados nas etapas posteriores do pipeline.


## 1. Validação dos arquivos de origem

Os arquivos CSV utilizados no projeto foram armazenados em um Volume gerenciado pelo Unity Catalog, localizado em `/Volumes/mvp_imobiliario/bronze/dados_origem`.

Antes da ingestão, é realizada uma verificação dos arquivos disponíveis no diretório para confirmar que todas as cinco fontes de dados estão acessíveis ao pipeline.

In [0]:
# Caminho do volume que contém os arquivos CSV originais
caminho_origem = "/Volumes/mvp_imobiliario/bronze/dados_origem"

# Lista os arquivos disponíveis no volume
arquivos = dbutils.fs.ls(caminho_origem)

for arquivo in arquivos:
    print(arquivo.name)

guru-dados-curitiba-pr.csv
guru-dados-recife-pe.csv
guru-dados-rio-de-janeiro-rj.csv
guru-dados-salvador-ba.csv
guru-dados-sao-paulo-sp.csv


## 2. Inspeção inicial da estrutura dos dados

Inicialmente, foi realizada a leitura de um dos arquivos para compreender sua estrutura e identificar o esquema dos dados.

Durante essa inspeção, verificou-se que o arquivo possui uma linha inicial contendo metadados sobre a fonte e a licença dos dados. Essa linha antecede o cabeçalho real do CSV e, portanto, exige tratamento específico durante a leitura.

In [0]:
# Leitura inicial de um arquivo para inspeção da estrutura dos dados
arquivo_teste = f"{caminho_origem}/guru-dados-salvador-ba.csv"

df_teste = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(arquivo_teste)
)

# Exibe a estrutura identificada pelo Spark
df_teste.printSchema()

# Exibe uma amostra dos dados
display(df_teste.limit(10))

root
 |-- # Guru dos Imóveis – dados abertos (CC-BY 4.0) · Salvador/BA · https://gurudosimoveis.com.br/dados: string (nullable = true)



# Guru dos Imóveis – dados abertos (CC-BY 4.0) · Salvador/BA · https://gurudosimoveis.com.br/dados
mes
2025-09
2025-09
2025-09
2025-09
2025-09
2025-09
2025-09
2025-09
2025-09


## 3. Diagnóstico do arquivo de origem

Para identificar a estrutura original do arquivo sem aplicar a interpretação automática de CSV, as primeiras linhas foram lidas como texto.

A inspeção confirmou que:

- a primeira linha contém informações sobre a fonte e a licença dos dados;
- a segunda linha contém o cabeçalho real;
- os registros iniciam a partir da terceira linha;
- o delimitador utilizado é a vírgula.

Com isso, definiu-se que as linhas iniciadas pelo caractere `#` devem ser ignoradas durante a leitura estruturada.

In [0]:
# Inspeção das primeiras linhas do arquivo original
linhas = spark.read.text(arquivo_teste)

display(linhas.limit(8))

value
# Guru dos Imóveis – dados abertos (CC-BY 4.0) · Salvador/BA · https://gurudosimoveis.com.br/dados
"mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio"
"2025-09,Acupe de Brotas,aluguel,,,,3,0,809"
"2025-09,Acupe de Brotas,venda,5084,3999,7357,19,0,554"
"2025-09,Águas Claras,venda,1600,1600,1600,1,0,2017"
"2025-09,ALPHAVILLE2,venda,5853,5829,6219,10,0,429"
"2025-09,Alphaville I,aluguel,67,62,77,37,3,254"
"2025-09,Alphaville I,venda,10798,9361,11935,213,20,230"


## 4. Validação da estratégia de leitura

Após o diagnóstico, foi realizada uma nova leitura utilizando o caractere `#` como indicador de comentário.

Essa configuração permite ignorar a linha inicial de metadados sem modificar os arquivos originais, preservando os dados recebidos da fonte e garantindo maior rastreabilidade do processo.

A leitura resultante identificou corretamente as nove colunas existentes nos arquivos.


In [0]:
# Leitura correta do CSV, ignorando a linha inicial de metadados
df_teste_corrigido = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .option("comment", "#")
    .option("sep", ",")
    .csv(arquivo_teste)
)

# Exibe o schema identificado
df_teste_corrigido.printSchema()

# Exibe uma amostra dos registros
display(df_teste_corrigido.limit(10))

root
 |-- mes: date (nullable = true)
 |-- bairro: string (nullable = true)
 |-- transacao: string (nullable = true)
 |-- mediana_m2: integer (nullable = true)
 |-- p25_m2: integer (nullable = true)
 |-- p75_m2: integer (nullable = true)
 |-- anuncios_ativos_media_dia: integer (nullable = true)
 |-- novos_anuncios: integer (nullable = true)
 |-- dias_no_mercado_medio: integer (nullable = true)



mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio
2025-09-01,Acupe de Brotas,aluguel,null,null,null,3,0,809
2025-09-01,Acupe de Brotas,venda,5084,3999,7357,19,0,554
2025-09-01,Águas Claras,venda,1600,1600,1600,1,0,2017
2025-09-01,ALPHAVILLE2,venda,5853,5829,6219,10,0,429
2025-09-01,Alphaville I,aluguel,67,62,77,37,3,254
2025-09-01,Alphaville I,venda,10798,9361,11935,213,20,230
2025-09-01,Alphaville II,aluguel,43,43,44,7,1,276
2025-09-01,Alphaville II,venda,7745,6644,8527,118,2,252
2025-09-01,Alto das Pombas,venda,5569,5569,5569,2,0,513
2025-09-01,Alto do Cabrito,venda,1633,1633,1633,1,0,1462


## 5. Ingestão consolidada na camada Bronze

Após validar a estrutura dos arquivos, foi definido explicitamente o schema utilizado na ingestão. Essa abordagem evita dependência da inferência automática de tipos e torna o pipeline mais previsível e reproduzível.

Os cinco arquivos são processados de forma conjunta e recebem metadados adicionais de rastreabilidade:

- `arquivo_origem`: identifica o arquivo CSV de origem;
- `cidade`: identifica a cidade correspondente ao arquivo;
- `uf`: identifica a Unidade Federativa;
- `data_ingestao`: registra a data e hora de processamento.

Nesta etapa não são realizados tratamentos de qualidade nos dados de negócio. Valores ausentes e demais características da fonte são preservados para posterior tratamento na camada Silver.

In [0]:
from pyspark.sql.types import (
    StructType, StructField, DateType,
    StringType, IntegerType
)

from pyspark.sql.functions import (
    col, regexp_extract, current_timestamp
)

# Schema explícito dos arquivos de origem
schema_bronze = StructType([
    StructField("mes", DateType(), True),
    StructField("bairro", StringType(), True),
    StructField("transacao", StringType(), True),
    StructField("mediana_m2", IntegerType(), True),
    StructField("p25_m2", IntegerType(), True),
    StructField("p75_m2", IntegerType(), True),
    StructField("anuncios_ativos_media_dia", IntegerType(), True),
    StructField("novos_anuncios", IntegerType(), True),
    StructField("dias_no_mercado_medio", IntegerType(), True)
])

# Leitura dos cinco arquivos CSV
df_bronze = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("comment", "#")
    .option("sep", ",")
    .schema(schema_bronze)
    .csv(f"{caminho_origem}/*.csv")

    # Metadados de rastreabilidade
    .withColumn("arquivo_origem", col("_metadata.file_path"))

    # Cidade extraída do nome do arquivo
    .withColumn(
        "cidade",
        regexp_extract(
            col("_metadata.file_path"),
            r"guru-dados-(.*)-[a-z]{2}\.csv$",
            1
        )
    )

    # UF extraída do nome do arquivo
    .withColumn(
        "uf",
        regexp_extract(
            col("_metadata.file_path"),
            r"-([a-z]{2})\.csv$",
            1
        )
    )

    # Data/hora em que ocorreu a ingestão
    .withColumn(
        "data_ingestao",
        current_timestamp()
    )
)

display(df_bronze.limit(10))

mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio,arquivo_origem,cidade,uf,data_ingestao
2025-09-01,Aclimação,aluguel,53,42,58,38,0,2135,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Aclimação,venda,12496,8314,14739,500,3,1230,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Aeroporto,venda,6719,6719,6719,1,0,2469,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Branca,aluguel,54,44,72,32,2,221,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Branca,venda,11153,9210,13960,199,1,423,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Fria,aluguel,39,33,50,13,0,443,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Fria,venda,7055,5895,8516,183,0,848,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Funda,aluguel,18,16,19,3,0,1899,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Funda,venda,5839,5086,7270,12,0,575,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z
2025-09-01,Água Rasa,aluguel,83,83,83,4,0,91,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-sao-paulo-sp.csv,sao-paulo,sp,2026-09-22T01:30:22.827Z


## 6. Validação da ingestão

Após a consolidação dos arquivos, é realizada uma validação da quantidade de registros por cidade e Unidade Federativa.

O objetivo desta etapa é verificar se todas as cinco fontes foram incorporadas ao DataFrame antes da persistência definitiva na camada Bronze.

In [0]:
from pyspark.sql.functions import count

# Validação da ingestão por cidade e UF
df_validacao = (
    df_bronze
    .groupBy("cidade", "uf")
    .agg(count("*").alias("quantidade_registros"))
    .orderBy("cidade")
)

display(df_validacao)

# Quantidade total de registros ingeridos
total_registros = df_bronze.count()

print(f"Total de registros ingeridos: {total_registros}")

cidade,uf,quantidade_registros
curitiba,pr,1797
recife,pe,1595
rio-de-janeiro,rj,2975
salvador,ba,3167
sao-paulo,sp,27399


Total de registros ingeridos: 36933


## 7. Persistência dos dados na camada Bronze

Após a validação da ingestão, os dados consolidados são persistidos em formato Delta no Unity Catalog.

A tabela `mvp_imobiliario.bronze.mercado_imobiliario_raw` representa a camada Bronze do pipeline e mantém os dados provenientes das fontes originais, acrescidos apenas dos metadados necessários para rastreabilidade.

O modo `overwrite` é utilizado para permitir a reexecução controlada do notebook durante o desenvolvimento, evitando a duplicação dos registros.

In [0]:
# Gravação dos dados consolidados na camada Bronze em formato Delta

tabela_bronze = "mvp_imobiliario.bronze.mercado_imobiliario_raw"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_bronze)
)

print(f"Tabela criada com sucesso: {tabela_bronze}")

Tabela criada com sucesso: mvp_imobiliario.bronze.mercado_imobiliario_raw


## 8. Validação da tabela persistida

Após a gravação, a tabela Delta é novamente consultada diretamente no Unity Catalog.

Essa validação tem como objetivo confirmar que a quantidade total de registros e a distribuição por cidade e UF permanecem consistentes após a persistência dos dados na camada Bronze.

In [0]:
# Validação da tabela Delta gravada no Unity Catalog

df_bronze_gravado = spark.table(
    "mvp_imobiliario.bronze.mercado_imobiliario_raw"
)

print(
    f"Quantidade de registros gravados: "
    f"{df_bronze_gravado.count()}"
)

display(
    df_bronze_gravado
    .groupBy("cidade", "uf")
    .count()
    .orderBy("cidade")
)

Quantidade de registros gravados: 36933


cidade,uf,count
curitiba,pr,1797
recife,pe,1595
rio-de-janeiro,rj,2975
salvador,ba,3167
sao-paulo,sp,27399


## Resultado da etapa Bronze

A ingestão consolidou com sucesso os dados das cinco cidades analisadas, totalizando **36.933 registros**.

Os arquivos originais foram preservados no Volume do Unity Catalog e os dados foram armazenados em formato Delta na tabela `mvp_imobiliario.bronze.mercado_imobiliario_raw`.

A camada Bronze mantém os dados sem tratamentos relacionados à qualidade das informações de negócio. Valores ausentes, possíveis inconsistências, duplicidades e demais aspectos de qualidade serão avaliados nas etapas seguintes do pipeline.

Com isso, a etapa de ingestão é concluída e os dados estão disponíveis para os processos de limpeza, padronização e transformação da camada Silver.